# 2018

In [1]:
# =========================
# 0. Imports & setup
# =========================
import warnings
import os
import re

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

pd.options.display.float_format = "{:.2f}".format
pd.set_option("display.max_colwidth", None)
warnings.filterwarnings("ignore")

os.chdir("/Users/nataschajademinnitt/Documents/5_data/teenage_pregnancy")
print("Current directory:", os.getcwd())

SURVEY_YEAR = 2018


# =========================
# 1. Load raw 2018 data
# =========================
df_load = pd.read_stata("./data/AGYW_dataset_2018.dta")


# =========================
# 2. Rename variables to canonical names
# =========================
rename_map_2018 = {
    # Interview context / schooling
    "Scol_status": "scol_status",
    "Scol_level": "scol_level",
    "Born_month": "born_month",
    "Born_year": "born_year",
    "Age_completed": "age_completed",
    "Attend_scol": "attend_scol",
    "Level_scol": "level_scol",
    "Left_scol": "left_scol",

    # 104c reasons
    "Lack_fees": "lack_fees",
    "Got_preg": "got_preg",
    "Got_married": "got_married",
    "Got_sick": "got_sick",
    "Need_money": "need_money",
    "Good_std": "good_std",
    "Int_scol": "int_scol",
    "Other_reas": "other_reas",

    # Literacy + marriage
    "Read_write": "read_write",
    "Current_married": "current_married",
    "More_wife": "more_wife",
    "Part_age": "part_age",

    # Assets
    "Radio": "radio",
    "Tv_set": "tv_set",
    "Bicycle": "bicycle",
    "Motorcycle": "motorcycle",
    "Own_home": "own_home",
    "Cell_phone": "cell_phone",
    "Reg_phone": "reg_phone",
    "Computer": "computer",
    "Income_busin": "income_busin",
    "Bath_room": "bath_room",
    "Run_water": "run_water",
    "electricity": "electricity",
    "car": "car",
    "generator": "generator",
    "solar": "solar",

    # Ever had sex & first sex
    "Life_sex": "life_sex",
    "Sex_age": "sex_age",
    "Person_sex": "person_sex",
    "Will_sex": "will_sex",

    # Did anything to prevent pregnancy at first sex
    "Do_anything": "do_anything",

    # 9 methods at first sex
    "Male_condom": "male_condom",
    "pill": "pill",
    "Injection": "injection",
    "Female_condom": "female_condom",
    "withdrawal": "withdrawal",
    "Emergency": "emergency",
    "Iud_coil": "iud_coil",
    "implant": "implant",
    "Avoid_other": "avoid_other",

    # Follow-up sex/condom questions
    "Last_sex": "last_sex",
    "Often_usecondom": "often_usecondom",
    "Relate_sex": "relate_sex",
    "Old_parner": "old_parner",
    "Partuse_condom": "partuse_condom",
    "Some_times": "some_times",
    "Worry_preg": "worry_preg",
    "Under_influe": "under_influe",

    # Pregnancy
    "Been_preg": "been_preg",
    "Age_preg": "age_preg",
    "Preg_end": "preg_end",
    "Current_use": "current_use",

    # Optional derived
    "educattained": "educattained",
    "married_relshp": "married_relshp",
    "Exper_sexualint": "exper_sexualint",
}

raw_cols = set(df_load.columns)  # BEFORE rename
df_load = df_load.rename(columns=rename_map_2018)

present = [k for k in rename_map_2018 if k in raw_cols]
missing = [k for k in rename_map_2018 if k not in raw_cols]
print("present:", len(present))
print("missing:", len(missing))
print("missing sample:", missing[:30])

# Quick method counts in renamed-but-not-lowercased df_load
method_cols = [
    "male_condom", "pill", "injection", "female_condom",
    "withdrawal", "emergency", "iud_coil", "implant", "avoid_other"
]
for c in method_cols:
    if c in df_load.columns:
        print(f"\n==== {c} ====")
        print(df_load[c].value_counts(dropna=False))

# Standardise colnames (do this BEFORE dropping by lower-case names)
df_load.columns = df_load.columns.str.lower()

# Drop nuisance / duplicate columns if they exist (now safe)
drop_if_exists = [
    "qn_111_do_you_or_do_ld_own_the_f",
    "sex_age_raw",
]
df_load = df_load.drop(columns=[c for c in drop_if_exists if c in df_load.columns], errors="ignore")

# Drop all-missing columns and restrict to rows where ever-preg question exists
df_load = df_load.dropna(axis=1, how="all")
df_load = df_load.dropna(subset=["been_preg"])
print("Raw 2018 after basic filter:", df_load.shape)


# =========================
# 3. Start cleaned copy
# =========================
df_clean = df_load.copy()


# =========================
# 4. Pregnancy & sex basics
# =========================
def recode_yes_no(series: pd.Series) -> pd.Series:
    s = series.copy()

    # Handle Stata labelled types that come in as object/category/string
    if (pd.api.types.is_object_dtype(s)
        or pd.api.types.is_categorical_dtype(s)
        or pd.api.types.is_string_dtype(s)):
        s2 = (
            s.astype("string").str.strip().str.lower()
            .replace({"nan": pd.NA, "": pd.NA})
        )
        return s2.map({"yes": 1, "y": 1, "no": 0, "n": 0}).astype("Int64")

    # Numeric coding: 1=yes, 2=no
    s_num = pd.to_numeric(s, errors="coerce")
    return s_num.map({1: 1, 2: 0}).astype("Int64")

df_clean["been_preg"] = recode_yes_no(df_clean["been_preg"])
print(df_clean["been_preg"].value_counts(dropna=False))

# Ever had sex (2018 likely 1/2)
life = df_clean["life_sex"]
life_num = pd.to_numeric(life, errors="coerce")
life_bin = life_num.map({1: 1, 2: 0}).astype("Int64")

# fallback if it was strings
if life_bin.isna().all():
    life_bin = (
        life.astype("string").str.strip().str.lower()
        .map({"yes": 1, "no": 0})
        .astype("Int64")
    )

df_clean["life_sex"] = life_bin
df_clean["age_completed"] = pd.to_numeric(df_clean["age_completed"], errors="coerce")

# Willingness: 1/2 willing, 3/4 not
df_clean["will_sex_binary"] = df_clean["will_sex"].map({1.0: 1, 2.0: 1, 3.0: 0, 4.0: 0}).astype("Int64")

# Did anything to prevent pregnancy at first sex
s = df_clean["do_anything"]
if pd.api.types.is_numeric_dtype(s):
    df_clean["do_anything_binary"] = (
        pd.to_numeric(s, errors="coerce")
        .map({1: 1.0, 2: 0.0, 3: np.nan})
        .astype("Float64")
    )
else:
    ss = s.astype("string").str.strip().str.lower()
    df_clean["do_anything_binary"] = (
        ss.map({"yes": 1.0, "no": 0.0, "don't remember": np.nan, "dont remember": np.nan})
        .astype("Float64")
    )

# Not asked / not applicable for never-sex
df_clean.loc[df_clean["life_sex"].eq(0), "do_anything_binary"] = pd.NA

print(df_clean["do_anything"].value_counts(dropna=False))
print(df_clean["do_anything_binary"].value_counts(dropna=False))
print(pd.crosstab(df_clean["do_anything_binary"], df_clean["life_sex"], dropna=False))


# =========================
# 5. First partner category — FIXED for 2018 numeric codes
# =========================
def recode_person_sex_group(series: pd.Series) -> pd.Categorical:
    num = pd.to_numeric(series, errors="coerce")
    out = pd.Series(pd.NA, index=series.index, dtype="object")

    if num.notna().sum() > 0:
        out.loc[num.eq(1)] = "boyfriend"
        out.loc[num.eq(2)] = "husband"
        out.loc[num.notna() & ~num.isin([1, 2])] = "other"
    else:
        s = series.astype("string").str.strip().str.lower()
        out.loc[s.str.contains("boyfriend", na=False)] = "boyfriend"
        out.loc[s.str.contains("husband", na=False)] = "husband"
        out.loc[s.notna() & out.isna()] = "other"

    return pd.Categorical(out, categories=["boyfriend", "husband", "other"], ordered=False)

df_clean["person_sex_group"] = recode_person_sex_group(df_clean["person_sex"])


# =========================
# 6. Methods at first sex — robust coding to 1/2/NA
# =========================
method_cols = [
    "male_condom", "female_condom", "iud_coil", "avoid_other",
    "pill", "withdrawal", "implant", "injection", "emergency"
]

life_num = pd.to_numeric(df_clean["life_sex"], errors="coerce")
sex_mask = life_num.eq(1)

for col in method_cols:
    raw = df_clean[col]
    v = pd.Series(pd.NA, index=df_clean.index, dtype="Float64")

    raw_num = pd.to_numeric(raw, errors="coerce")
    v.loc[sex_mask & raw_num.eq(1)] = 1.0
    v.loc[sex_mask & raw_num.eq(2)] = 2.0

    raw_str = raw.astype("string").str.strip().str.lower()
    miss = v.isna()
    v.loc[miss & sex_mask & raw_str.eq("selected")] = 1.0
    v.loc[miss & sex_mask & raw_str.eq("not selected")] = 2.0

    df_clean[col] = v

print("male_condom AFTER step6:")
print(df_clean["male_condom"].value_counts(dropna=False).head(10))


# =========================
# 7. Condom use in last 12 months — often_usecondom
# =========================
ocon = pd.to_numeric(df_clean["often_usecondom"], errors="coerce")
df_clean["sex_active_12m"] = ocon.isin([1, 2, 3]).astype(int)
cond_map = {1: 0, 2: 1, 3: 2}
df_clean["condom_use_ord"] = ocon.map(cond_map)
df_clean["condom_use_ord_active"] = df_clean["condom_use_ord"].fillna(0) * df_clean["sex_active_12m"]
df_clean["often_usecondom"] = ocon


# =========================
# 8. Schooling: dropout reasons + level recode + class_clean + marital binaries
# =========================
df_clean["scol_status"] = df_clean["scol_status"].astype("string").str.strip().str.upper()

df_clean.loc[
    (df_clean["scol_status"] == "OUT OF SCHOOL") & df_clean["attend_scol"].isna(),
    "attend_scol"
] = "No"

att = df_clean["attend_scol"].astype("string").str.strip().str.title()
df_clean["attend_scol_binary"] = att.map({"Yes": 1, "No": 0}).astype("Int64")

drop_cols = ["lack_fees", "got_preg", "got_married", "got_sick", "need_money", "good_std", "int_scol"]
for col in drop_cols:
    s = df_clean[col].astype("string").str.strip().str.lower()
    df_clean[col] = s.eq("selected").fillna(False).astype("int8")

def norm_level_str(x):
    if pd.isna(x):
        return np.nan
    s = str(x).upper()
    s = (s.replace("’", "'").replace("‘", "'").replace("–", "-").replace("—", "-"))
    s = (s.replace("\u00A0", " ").replace("\u202F", " ").replace("\u2009", " "))
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"\s*-\s*", " - ", s)
    s = re.sub(r"\bO\s*LEVEL\b", "O' LEVEL", s)
    s = re.sub(r"\bA\s*LEVEL\b", "A' LEVEL", s)
    s = re.sub(r"\bP\s*\.?\s*([1-7])\b", r"P.\1", s)
    s = re.sub(r"\bS\s*\.?\s*([1-6])\b", r"S.\1", s)
    s = re.sub(r"(P\.[1-7])\s*-\s*(P\.[1-7])", r"\1 - \2", s)
    s = re.sub(r"(S\.[1-6])\s*-\s*(S\.[1-6])", r"\1 - \2", s)
    if s == "OTHER":
        s = "OTHER (SPECIFY)"
    return s

lvl_norm = df_clean["level_scol"].apply(norm_level_str)
codes = pd.Series(pd.NA, index=df_clean.index, dtype="Int64")

exact_map = {
    "PRIMARY (P.1 - P.7)": 2,
    "PRIMARY PROFESSIONAL": 3,
    "O' LEVEL (S.1 - S.4)": 4,
    "O' LEVEL PROFESSIONAL": 5,
    "A' LEVEL (S.5 - S.6)": 6,
    "UNIVERSITY": 7,
    "OTHER TERTIARY (AFTER S.6)": 8,
    "OTHER (SPECIFY)": 9,
}
for key, val in exact_map.items():
    codes[lvl_norm == key] = val

need = codes.isna()
ln = lvl_norm[need]
codes.loc[need & ln.str.contains(r"\bUNIVERSITY\b", na=False)] = 7
codes.loc[need & ln.str.contains(r"\b(COLLEGE|TERTIARY|INSTITUTE|POLYTECH|DIPLOMA|DEGREE)\b", na=False)] = 8
codes.loc[need & ln.str.contains(r"\bA' LEVEL\b|\bS\.[5-6]\b|\bFORM\s*[5-6]\b", na=False)] = 6
codes.loc[need & ln.str.contains(r"\bO' LEVEL\b|\bS\.[1-4]\b|\bFORM\s*[1-4]\b", na=False)] = 4
codes.loc[need & ln.str.contains(r"O' LEVEL.*PROF|PROFESSIONAL", na=False)] = 5
codes.loc[need & ln.str.contains(r"\bPRIMARY\b|\bPRI\b|\bP\.[1-7]\b", na=False)] = 2
codes.loc[need & ln.str.contains(r"PRIMARY.*PROF|PROFESSIONAL", na=False)] = 3
df_clean["level_scol_recode"] = codes

class_cats = [*(f"P{i}" for i in range(1, 8)), *(f"S{i}" for i in range(1, 7)), "88", "89"]
if "class" in df_clean.columns:
    s = df_clean["class"].astype(str).str.strip().str.upper()
    s = s.str.replace("’", "'", regex=False).str.replace("–", "-", regex=False).str.replace("—", "-", regex=False)
    s = s.replace({"": np.nan, "NAN": np.nan, "NA": np.nan, "NONE": np.nan})
    norm = s.str.replace(r"[^A-Z0-9]", "", regex=True)
    norm = norm.str.replace(r"^UNIVERSITY(?:88)?$", "88", regex=True)
    norm = norm.str.replace(r"^OTHERTERTIARY(?:89)?$", "89", regex=True)
    norm = norm.str.replace(r"^Y\d+$", "88", regex=True)
    norm = norm.str.replace(r"^J([1-3])$", r"S\1", regex=True)
    norm = norm.str.replace(r"^5([1-6])$", r"S\1", regex=True)
    norm = norm.mask(norm.eq("S"))
    norm = norm.str.replace(r"^98$", "89", regex=True)
    m_bare = norm.str.fullmatch(r"[1-7]").fillna(False)
    norm.loc[m_bare] = "P" + norm.loc[m_bare]
    is_pure_digit = norm.str.fullmatch(r"\d+")
    norm = norm.mask(is_pure_digit & ~norm.isin({"88", "89"}))
    df_clean["class_clean"] = pd.Categorical(norm.where(norm.isin(set(class_cats))), categories=class_cats)
else:
    df_clean["class_clean"] = pd.Categorical([pd.NA] * len(df_clean), categories=class_cats)

df_clean["current_married_binary"] = df_clean["current_married"].map({
    "MARRIED/UNION": 1,
    "DIVORCED/SEPARATED": 0,
    "WIDOWED": 0,
    "NEVER MARRIED": 0,
    "IN RELATIONSHIP BUT NOT MARRIED": 0,
})
df_clean["been_married_binary"] = df_clean["current_married"].map({
    "MARRIED/UNION": 1,
    "DIVORCED/SEPARATED": 1,
    "WIDOWED": 1,
    "NEVER MARRIED": 0,
    "IN RELATIONSHIP BUT NOT MARRIED": 0,
})

df_clean["age_marry"] = pd.to_numeric(df_clean.get("age_marry"), errors="coerce")
q1, q3 = df_clean["age_marry"].quantile(0.25), df_clean["age_marry"].quantile(0.75)
iqr = q3 - q1
df_clean["age_marry"] = df_clean["age_marry"].clip(lower=(q1 - 1.5 * iqr))
df_clean["married_by19"] = (df_clean["age_marry"] <= 19).astype(float)


# =========================
# 9. Birth year/month & pregnancy timing
# =========================
df_clean.loc[df_clean["born_year"].isna(), "born_year"] = SURVEY_YEAR - df_clean["age_completed"]
df_clean.loc[df_clean["born_month"].isna(), "born_month"] = 7

df_clean["born_month"] = pd.to_numeric(df_clean["born_month"], errors="coerce")
df_clean["born_year"] = pd.to_numeric(df_clean["born_year"], errors="coerce")

df_clean.loc[~df_clean["born_month"].between(1, 12), "born_month"] = np.nan
df_clean.loc[~df_clean["born_year"].between(SURVEY_YEAR - 24, SURVEY_YEAR - 10), "born_year"] = np.nan
df_clean["born_month"] = df_clean["born_month"].fillna(7)

df_clean["age_months"] = (SURVEY_YEAR - df_clean["born_year"]) * 12 + (7 - df_clean["born_month"])

df_clean["age_preg"] = pd.to_numeric(df_clean["age_preg"], errors="coerce")
df_clean.loc[~df_clean["age_preg"].between(10, 24), "age_preg"] = np.nan
df_clean.loc[(df_clean["been_preg"] == 1) & (df_clean["age_preg"].isna()), "age_preg"] = df_clean["age_completed"] - 1


# =========================
# 10. Harmonise sex_age, age_preg, age_completed, age_marry (single-pass + requested fix)
# =========================
df_clean["sex_age"] = pd.to_numeric(df_clean["sex_age"], errors="coerce")

# if been_preg==0 but age_preg present -> set been_preg=1
df_clean.loc[(df_clean["been_preg"] == 0) & (df_clean["age_preg"].notna()), "been_preg"] = 1

# bounds
df_clean.loc[~df_clean["sex_age"].between(5, 24), "sex_age"] = np.nan

# if pregnant and sex_age missing -> sex_age = age_preg
df_clean.loc[(df_clean["been_preg"] == 1) & (df_clean["sex_age"].isna()), "sex_age"] = df_clean["age_preg"]

# requested: if sex_age > age_preg -> set sex_age = age_preg
mask_fix = (df_clean["been_preg"] == 1) & df_clean["sex_age"].notna() & df_clean["age_preg"].notna()
df_clean.loc[mask_fix & (df_clean["sex_age"] > df_clean["age_preg"]), "sex_age"] = df_clean["age_preg"]

# ensure age_preg <= age_completed
df_clean.loc[(df_clean["been_preg"] == 1) & (df_clean["age_preg"] > df_clean["age_completed"]), "age_preg"] = df_clean["age_completed"]

# ensure age_marry <= age_completed
df_clean.loc[df_clean["age_marry"] > df_clean["age_completed"], "age_marry"] = df_clean["age_completed"]

# if never married but age_marry present -> set NA
df_clean.loc[(df_clean["been_married_binary"] == 0) & df_clean["age_marry"].notna(), "age_marry"] = np.nan

# remove implausible
df_clean = df_clean[(df_clean["age_preg"].isna()) | (df_clean["age_preg"] <= 25)].copy()

df_clean.loc[df_clean["been_preg"] == 1, "diff"] = (df_clean["age_months"] / 12 - df_clean["age_preg"])
df_clean.loc[df_clean["diff"] < -0.5, "diff"] = np.nan

df_clean["ado_preg"] = ((df_clean["been_preg"] == 1) & (df_clean["age_preg"] <= 19)).fillna(False).astype("int8")

mask_fix = (
    (df_clean["been_preg"] == 1)
    & df_clean["sex_age"].notna()
    & df_clean["age_preg"].notna()
    & (df_clean["sex_age"] > df_clean["age_preg"])
)
df_clean.loc[mask_fix, "sex_age"] = df_clean["age_preg"]



# =========================
# 11. Marriage timing categories
# =========================
def marriage_any_vs_cases(row):
    ap = pd.to_numeric(row["age_preg"], errors="coerce")
    am = pd.to_numeric(row["age_marry"], errors="coerce")
    if pd.isna(ap) and pd.isna(am):
        return "never_marry_or_preg"
    if pd.notna(ap) and pd.isna(am):
        return "preg_never_marry"
    if pd.isna(ap) and pd.notna(am):
        return "marry_never_preg"
    return "marry_after_preg" if am > ap else "marry_before_preg"

df_clean["marriage_timing"] = df_clean.apply(marriage_any_vs_cases, axis=1)


# =========================
# 12. Years of schooling & buckets
# =========================
years_from_class = {**{f"P{i}": i - 1 for i in range(1, 8)}, "S1": 7, "S2": 8, "S3": 9, "S4": 10, "S5": 11, "S6": 12, "88": 13, "89": 13}
cls = df_clean["class_clean"].astype("string").str.strip()
years_cls = cls.map(years_from_class).astype("float")

level = pd.to_numeric(df_clean["level_scol_recode"], errors="coerce")
years_from_level = pd.Series(np.nan, index=df_clean.index, dtype="float")
years_from_level[level.isin([2, 3])] = 7
years_from_level[level.isin([4, 5])] = 11
years_from_level[level.eq(6)] = 13
years_from_level[level.isin([7, 8, 9])] = 14

df_clean["years_school"] = pd.concat([years_cls, years_from_level], axis=1).max(axis=1).fillna(0).clip(lower=0)

class_to_band = {**{f"P{i}": "less_than_UPE" for i in range(1, 8)}, **{f"S{i}": "UPE" for i in range(1, 5)}, **{f"S{i}": "USE" for i in range(5, 7)}, "88": "higher_than_USE", "89": "higher_than_USE"}
from_class_band = df_clean["class_clean"].map(class_to_band)

level_band_map = {2: "UPE", 3: "UPE", 4: "USE", 5: "USE", 6: "higher_than_USE", 7: "higher_than_USE", 8: "higher_than_USE", 9: "higher_than_USE"}
from_level_band = level.map(level_band_map)

attend_norm = df_clean["attend_scol"].astype("string").str.strip().str.lower() if "attend_scol" in df_clean.columns else pd.Series(False, index=df_clean.index)
never_attended = attend_norm.isin(["no", "0", "false"]) if isinstance(attend_norm, pd.Series) else pd.Series(False, index=df_clean.index)

status_norm = df_clean["scol_status"].astype("string").str.strip().str.lower().where(lambda x: x.isin(["in school", "out of school"]), other=pd.NA)
is_in = status_norm.eq("in school")
is_out = status_norm.eq("out of school")

edu_high = pd.Series(pd.NA, index=df_clean.index, dtype="object")

idx = is_in.fillna(False)
edu_high.loc[idx] = from_class_band.loc[idx]
miss = idx & edu_high.isna() & from_level_band.notna()
edu_high.loc[miss] = from_level_band.loc[miss]

idx = is_out.fillna(False)
edu_high.loc[idx] = from_level_band.loc[idx]
miss = idx & edu_high.isna() & from_class_band.notna()
edu_high.loc[miss] = from_class_band.loc[miss]

idx = ~(is_in.fillna(False) | is_out.fillna(False))
edu_high.loc[idx] = from_level_band.loc[idx]
miss = idx & edu_high.isna() & from_class_band.notna()
edu_high.loc[miss] = from_class_band.loc[miss]

edu_high.loc[never_attended] = "None"

cats = ["None", "less_than_UPE", "UPE", "USE", "higher_than_USE", "Unknown"]
df_clean["edu_bucket_highest"] = pd.Categorical(pd.Series(edu_high).fillna("Unknown"), categories=cats, ordered=True)

def make_school_complete3(df):
    status = df["scol_status"].astype("string").str.strip().str.lower().where(lambda x: x.isin(["in school", "out of school"]), other=pd.NA)
    ebh = df["edu_bucket_highest"].astype("string")
    out = pd.Series(pd.NA, index=df.index, dtype="Int64")
    out[status.eq("in school")] = 0
    out[status.eq("out of school") & ebh.isin(["USE", "higher_than_USE"])] = 1
    out[status.eq("out of school") & ebh.isin(["less_than_UPE", "UPE"])] = 2
    out[status.isna() & ebh.isin(["USE", "higher_than_USE"])] = 1
    out[status.isna() & ebh.isin(["less_than_UPE", "UPE"])] = 2
    return out

df_clean["school_complete3"] = make_school_complete3(df_clean)
label_map = {0: "in_school", 1: "completed_lower_secondary", 2: "dropped_out"}
df_clean["school_complete3_lbl"] = df_clean["school_complete3"].map(label_map).astype("string")


# =========================
# 13. Age cohort
# =========================
df_clean["age_cohort"] = np.where(df_clean["age_completed"] <= 14, "10–14", "15–19")


# =========================
# 14. Wealth PCA & tertiles
# =========================
asset_vars = ["radio", "tv_set", "bicycle", "motorcycle", "own_home", "cell_phone", "reg_phone", "computer", "income_busin", "bath_room", "run_water", "electricity", "car", "generator", "solar"]
df_assets = df_clean[asset_vars].replace(98, np.nan).copy()
df_assets = df_assets.applymap(lambda x: 1 if x == 1 else 0).fillna(0)

scaler = StandardScaler()
assets_scaled = scaler.fit_transform(df_assets)

pca = PCA(n_components=1)
wealth_index = pca.fit_transform(assets_scaled)
df_clean["wealth_index"] = wealth_index[:, 0]

w = df_clean["wealth_index"]
mask = w.notna()
ranks = w[mask].rank(method="first")
labels = ["Low", "Medium", "High"]
tertiles = pd.qcut(ranks, 3, labels=labels)

df_clean["wealth_tertile"] = pd.NA
df_clean.loc[mask, "wealth_tertile"] = tertiles.astype(str)
df_clean["wealth_tertile"] = pd.Categorical(df_clean["wealth_tertile"], categories=labels, ordered=True)


# =========================
# 15. Final export (aligned)
# =========================
FEATURES_2018 = [
    "ado_preg", "been_preg", "age_completed", "age_cohort", "age_preg", "sex_age",
    "will_sex_binary", "do_anything_binary", "age_marry", "married_by19", "marriage_timing",
    "school_complete3_lbl", "years_school", "level_scol_recode", "edu_bucket_highest",
    "person_sex_group", "male_condom", "female_condom", "iud_coil", "avoid_other",
    "pill", "withdrawal", "implant", "often_usecondom", "sex_active_12m",
    "condom_use_ord", "condom_use_ord_active", "wealth_tertile",
]

cols = [c for c in FEATURES_2018 if c in df_clean.columns]
df_export_2018 = df_clean[cols].copy()
df_export_2018["year"] = SURVEY_YEAR

df_export_2018.to_csv("./data/processed_df_2018_aligned.csv", index=False)
print("\nExported 2018 file with columns:")
print(df_export_2018.info())
print(df_export_2018.head())


Current directory: /Users/nataschajademinnitt/Documents/5_data/teenage_pregnancy
present: 64
missing: 0
missing sample: []

==== male_condom ====
male_condom
NaN     6186
1.00    1944
2.00     106
Name: count, dtype: int64

==== pill ====
pill
NaN     7246
2.00     906
1.00      84
Name: count, dtype: int64

==== injection ====
injection
NaN     7400
2.00     803
1.00      33
Name: count, dtype: int64

==== female_condom ====
female_condom
NaN     7415
2.00     806
1.00      15
Name: count, dtype: int64

==== withdrawal ====
withdrawal
NaN     7395
2.00     791
1.00      50
Name: count, dtype: int64

==== emergency ====
emergency
NaN     7413
2.00     797
1.00      26
Name: count, dtype: int64

==== iud_coil ====
iud_coil
NaN     7423
2.00     805
1.00       8
Name: count, dtype: int64

==== implant ====
implant
NaN     7430
2.00     797
1.00       9
Name: count, dtype: int64

==== avoid_other ====
avoid_other
NaN     7466
2.00     721
1.00      49
Name: count, dtype: int64
Raw 2018 af

# 2023

In [2]:
# =========================
# 0. Imports & setup
# =========================
import warnings
import os
import re

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

pd.options.display.float_format = "{:.2f}".format
pd.set_option("display.max_colwidth", None)
warnings.filterwarnings("ignore")

os.chdir("/Users/nataschajademinnitt/Documents/5_data/teenage_pregnancy")
print("Current directory:", os.getcwd())

SURVEY_YEAR = 2023


# =========================
# 1. Load raw 2023 data
# =========================
df_load = pd.read_stata("./data/AGYW_dataset_2023.dta")


# =========================
# 2. Rename variables to match 2018 naming
# =========================
rename_map = {
    # Interview context / schooling
    "SCHOOLING_STATUS": "scol_status",
    "SCHOOL_LEVEL": "scol_level",
    "QN_101_In_what_month_were_you": "born_month",
    "QN_101_In_what_year_were_you_": "born_year",
    "QN_102_How_old_were_EAR_OF_BIRTH": "age_completed",

    "education": "educattained",
    "marital": "married_relshp",
    "eversex": "exper_sexualint",
    "agefirstsex": "sex_age",

    "_103a_Have_you_ever_attended_s": "attend_scol",
    "q104a": "level_scol",  # <-- use this for schooling buckets (as you requested)
    "QN_104b_How_long_ha_ince_you_lef": "left_scol",

    # Dropout reasons
    "QN_104c_What_are_some_of_the_1": "lack_fees",
    "QN_104c_What_are_some_of_the_2": "got_preg",
    "QN_104c_What_are_some_of_the_3": "got_married",
    "QN_104c_What_are_some_of_the_4": "got_sick",
    "QN_104c_What_are_some_of_the_5": "need_money",
    "QN_104c_What_are_some_of_the_6": "good_std",
    "QN_104c_What_are_some_of_the_7": "int_scol",
    "QN_104c_What_are_some_of_the_8": "other_reas",

    "QN105a_Are_you_able_to_read_a": "read_write",
    "QN_107a_Are_you_currently_mar": "current_married",
    "QN_107b_Does_your_s_he_considers": "more_wife",
    "QN_107c_How_old_is_your_partner": "part_age",

    # Assets
    "A_radio": "radio",
    "A_television_set": "tv_set",
    "A_bicycle": "bicycle",
    "A_motor_cycle": "motorcycle",
    "Your_own_family_home": "own_home",
    "a_cell_phone": "cell_phone",
    "a_regular_land_line_phone": "reg_phone",
    "a_computer": "computer",
    "An_income_generating_business": "income_busin",
    "An_indoor_bathroom": "bath_room",
    "Running_water_either_mpound_of_y": "run_water",
    "Electricity": "electricity",
    "Car": "car",
    "Generator": "generator",
    "Solar": "solar",

    # Sex + first sex questions
    "QN_401_Have_you_ever_had_any_": "life_sex",
    "QN_202_How_old_were_the_very_fir": "sex_age_raw",  # drop; use agefirstsex -> sex_age
    "QN_403_Which_person_did_you_h": "person_sex",
    "QN_205_The_first_ti_r_not_willin": "will_sex",
    "QN_406_The_first_time_you_had": "do_anything",

    # Methods at first sex
    "QN_407_The_first_time_you_had": "method_raw",
    "QN_407_The_first_time_you_had1": "male_condom",
    "QN_407_The_first_time_you_had2": "pill",
    "QN_407_The_first_time_you_had3": "injection",
    "QN_407_The_first_time_you_had4": "female_condom",
    "QN_407_The_first_time_you_had5": "withdrawal",
    "QN_407_The_first_time_you_had6": "emergency",
    "QN_407_The_first_time_you_had7": "iud_coil",
    "QN_407_The_first_time_you_had8": "implant",
    "QN_407_The_first_time_you_had9": "avoid_other",

    # Follow-ups
    "QN_409_When_was_the_last_time": "last_sex",
    "QN_211_In_the_past_h_all_these_p": "often_usecondom",
    "QN_412_What_was_your_relation": "relate_sex",
    "QN_213_How_old_was_th_for_the_la": "old_parner",
    "QN_414_Thinking_of_THE_LAST_T": "partuse_condom",
    "QN_215_Thinking_of_sometimes_or_": "some_times",
    "QN_417_Are_you_currently_usin": "current_use",

    # Pregnancy
    "QN_1001_Have_you_ever_been_pr": "been_preg",
    "QN_702_At_what_age_t_for_the_fir": "age_preg",
    "QN_706_Are_you_currently_preg": "current_preg",
    "QN_103_How_did_the_first_preg": "preg_end",
}

df_load = df_load.rename(columns=rename_map)

# Drop ownership var if present
if "QN_111_Do_you_or_do_ld_own_the_f" in df_load.columns:
    df_load = df_load.drop(columns=["QN_111_Do_you_or_do_ld_own_the_f"])

# Drop raw first-sex age (we use agefirstsex -> sex_age)
if "sex_age_raw" in df_load.columns:
    df_load = df_load.drop(columns=["sex_age_raw"])

# Standardise column names
df_load.columns = df_load.columns.str.lower()

# Drop all-missing columns
df_load = df_load.dropna(axis=1, how="all")

# Restrict to rows where ever-pregnancy question exists (your 3307 subset)
df_load = df_load.dropna(subset=["been_preg"])
print("After basic filtering (ever-pregnancy asked):", df_load.shape)


# =========================
# 3. Start cleaned copy
# =========================
df_clean = df_load.copy()


# =========================
# 4. Pregnancy & sex basics
# =========================
df_clean["been_preg"] = (
    df_clean["been_preg"].astype("string").str.strip().str.title()
    .map({"Yes": 1, "No": 0})
    .astype("Int64")
)

df_clean["life_sex"] = (
    df_clean["life_sex"].astype("string").str.strip().str.lower()
    .map({"yes": 1, "no": 0})
    .astype("Int64")
)

print(df_clean["life_sex"].value_counts(dropna=False))

df_clean["age_completed"] = pd.to_numeric(df_clean["age_completed"], errors="coerce")

df_clean["will_sex_binary"] = df_clean["will_sex"].map({1.0: 1, 2.0: 1, 3.0: 0, 4.0: 0}).astype("Int64")

df_clean["do_anything_binary"] = (
    df_clean["do_anything"].astype("string").str.strip().str.lower()
    .replace({"dont remember": "don't remember"})
    .map({"yes": 1.0, "no": 0.0, "don't remember": np.nan})
    .astype("Float64")
)

df_clean.loc[df_clean["life_sex"].eq(0), "do_anything_binary"] = pd.NA


# =========================
# 5. First partner category (boyfriend / husband / other)
# =========================
def classify_partner_str(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if "boyfriend" in s:
        return "boyfriend"
    if "husband" in s:
        return "husband"
    return "other"

df_clean["person_sex_group"] = df_clean["person_sex"].apply(classify_partner_str)
df_clean["person_sex_group"] = pd.Categorical(
    df_clean["person_sex_group"],
    categories=["boyfriend", "husband", "other"],
    ordered=False,
)


# =========================
# 6. Contraceptive methods at first sex (1=used, 2=not used, NA=not asked/missing)
# =========================
method_cols = [
    "male_condom",
    "female_condom",
    "iud_coil",
    "avoid_other",
    "pill",
    "withdrawal",
    "implant",
    "injection",
    "emergency",
]

sex_mask = df_clean["life_sex"].eq(1)

for col in method_cols:
    s = df_clean[col].astype("string").str.strip().str.lower()
    v = pd.Series(pd.NA, index=df_clean.index, dtype="Float64")
    v.loc[sex_mask & s.eq("selected")] = 1.0
    v.loc[sex_mask & s.eq("not selected")] = 2.0
    df_clean[col] = v


# =========================
# 7. Condom use frequency in last 12 months
# =========================
ocon = pd.to_numeric(df_clean["often_usecondom"], errors="coerce")

df_clean["sex_active_12m"] = ocon.isin([1, 2, 3]).astype(int)
cond_map = {1: 0, 2: 1, 3: 2}
df_clean["condom_use_ord"] = ocon.map(cond_map)
df_clean["condom_use_ord_active"] = df_clean["condom_use_ord"].fillna(0) * df_clean["sex_active_12m"]
df_clean["often_usecondom"] = ocon


# =========================
# 8. Schooling: status normalisation + dropout reasons + level recode + edu buckets
# =========================
# Normalise SCHOOLING_STATUS so you get IN SCHOOL vs OUT OF SCHOOL (not "IN-SCHOOL")
status = df_clean["scol_status"].astype("string").str.strip().str.upper()
status = status.replace({
    "IN-SCHOOL": "IN SCHOOL",
    "IN SCHOOL": "IN SCHOOL",
    "OUT-OF-SCHOOL": "OUT OF SCHOOL",
    "OUT OF SCHOOL": "OUT OF SCHOOL",
})
df_clean["scol_status"] = status
print(df_clean["scol_status"].value_counts(dropna=False))

# If out-of-school and attend_scol is missing, set to "No"
df_clean.loc[(df_clean["scol_status"] == "OUT OF SCHOOL") & df_clean["attend_scol"].isna(), "attend_scol"] = "No"

att = df_clean["attend_scol"].astype("string").str.strip().str.title()
df_clean["attend_scol_binary"] = att.map({"Yes": 1, "No": 0}).astype("Int64")

drop_cols = ["lack_fees", "got_preg", "got_married", "got_sick", "need_money", "good_std", "int_scol"]
for col in drop_cols:
    s = df_clean[col].astype("string").str.strip().str.lower()
    df_clean[col] = s.eq("selected").fillna(False).astype("int8")

def norm_level_str(x):
    if pd.isna(x):
        return np.nan
    s = str(x).upper()
    s = (s.replace("’", "'").replace("‘", "'").replace("–", "-").replace("—", "-"))
    s = (s.replace("\u00A0", " ").replace("\u202F", " ").replace("\u2009", " "))
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"\s*-\s*", " - ", s)
    s = re.sub(r"\bO\s*LEVEL\b", "O' LEVEL", s)
    s = re.sub(r"\bA\s*LEVEL\b", "A' LEVEL", s)
    s = re.sub(r"\bP\s*\.?\s*([1-7])\b", r"P.\1", s)
    s = re.sub(r"\bS\s*\.?\s*([1-6])\b", r"S.\1", s)
    s = re.sub(r"(P\.[1-7])\s*-\s*(P\.[1-7])", r"\1 - \2", s)
    s = re.sub(r"(S\.[1-6])\s*-\s*(S\.[1-6])", r"\1 - \2", s)
    if s == "OTHER":
        s = "OTHER (SPECIFY)"
    return s

lvl_norm = df_clean["level_scol"].apply(norm_level_str)

codes = pd.Series(pd.NA, index=df_clean.index, dtype="Int64")
exact_map = {
    "PRIMARY (P.1 - P.7)": 2,
    "PRIMARY PROFESSIONAL": 3,
    "O' LEVEL (S.1 - S.4)": 4,
    "O' LEVEL PROFESSIONAL": 5,
    "A' LEVEL (S.5 - S.6)": 6,
    "UNIVERSITY": 7,
    "OTHER TERTIARY (AFTER S.6)": 8,
    "OTHER (SPECIFY)": 9,
}
for key, val in exact_map.items():
    codes[lvl_norm == key] = val

need = codes.isna()
ln = lvl_norm[need]
codes.loc[need & ln.str.contains(r"\bUNIVERSITY\b", na=False)] = 7
codes.loc[need & ln.str.contains(r"\b(COLLEGE|TERTIARY|INSTITUTE|POLYTECH|DIPLOMA|DEGREE)\b", na=False)] = 8
codes.loc[need & ln.str.contains(r"\bA' LEVEL\b|\bS\.[5-6]\b|\bFORM\s*[5-6]\b", na=False)] = 6
codes.loc[need & ln.str.contains(r"\bO' LEVEL\b|\bS\.[1-4]\b|\bFORM\s*[1-4]\b", na=False)] = 4
codes.loc[need & ln.str.contains(r"O' LEVEL.*PROF|PROFESSIONAL", na=False)] = 5
codes.loc[need & ln.str.contains(r"\bPRIMARY\b|\bPRI\b|\bP\.[1-7]\b", na=False)] = 2
codes.loc[need & ln.str.contains(r"PRIMARY.*PROF|PROFESSIONAL", na=False)] = 3

df_clean["level_scol_recode"] = codes

# No class variable in 2023 but keep structure
class_cats = [*(f"P{i}" for i in range(1, 8)), *(f"S{i}" for i in range(1, 7)), "88", "89"]
df_clean["class_clean"] = pd.Categorical([pd.NA] * len(df_clean), categories=class_cats)

# Years of school (from level only in 2023)
years_from_level = pd.Series(np.nan, index=df_clean.index, dtype="float")
lvl = pd.to_numeric(df_clean["level_scol_recode"], errors="coerce")
years_from_level[lvl.isin([2, 3])] = 7
years_from_level[lvl.isin([4, 5])] = 11
years_from_level[lvl.eq(6)] = 13
years_from_level[lvl.isin([7, 8, 9])] = 14
df_clean["years_school"] = years_from_level.fillna(0).clip(lower=0)

# edu_bucket_highest (from level + schooling status)
level_band_map = {
    2: "UPE", 3: "UPE",
    4: "USE", 5: "USE",
    6: "higher_than_USE", 7: "higher_than_USE",
    8: "higher_than_USE", 9: "higher_than_USE"
}
from_level_band = lvl.map(level_band_map)

attend_norm = df_clean["attend_scol"].astype("string").str.strip().str.lower() if "attend_scol" in df_clean.columns else pd.Series(False, index=df_clean.index)
never_attended = attend_norm.isin(["no", "0", "false"]) if isinstance(attend_norm, pd.Series) else pd.Series(False, index=df_clean.index)

status_norm = df_clean["scol_status"].astype("string").str.strip().str.lower().where(lambda x: x.isin(["in school", "out of school"]), other=pd.NA)
is_in = status_norm.eq("in school")
is_out = status_norm.eq("out of school")

edu_high = pd.Series(pd.NA, index=df_clean.index, dtype="object")

# In school / out of school both use from_level_band in 2023
edu_high.loc[is_in.fillna(False)] = from_level_band.loc[is_in.fillna(False)]
edu_high.loc[is_out.fillna(False)] = from_level_band.loc[is_out.fillna(False)]
edu_high.loc[edu_high.isna()] = from_level_band.loc[edu_high.isna()]

edu_high.loc[never_attended] = "None"

cats = ["None", "less_than_UPE", "UPE", "USE", "higher_than_USE", "Unknown"]
edu_high = edu_high.replace({pd.NA: "Unknown"}).fillna("Unknown")

# If level_scol is missing, we can't really separate less_than_UPE vs UPE in 2023 → keep Unknown
df_clean["edu_bucket_highest"] = pd.Categorical(edu_high, categories=cats, ordered=True)

def make_school_complete3(df):
    status = df["scol_status"].astype("string").str.strip().str.lower().where(lambda x: x.isin(["in school", "out of school"]), other=pd.NA)
    ebh = df["edu_bucket_highest"].astype("string")

    out = pd.Series(pd.NA, index=df.index, dtype="Int64")
    out[status.eq("in school")] = 0
    out[status.eq("out of school") & ebh.isin(["USE", "higher_than_USE"])] = 1
    out[status.eq("out of school") & ebh.isin(["less_than_UPE", "UPE"])] = 2
    out[status.isna() & ebh.isin(["USE", "higher_than_USE"])] = 1
    out[status.isna() & ebh.isin(["less_than_UPE", "UPE"])] = 2
    return out

df_clean["school_complete3"] = make_school_complete3(df_clean)
label_map = {0: "in_school", 1: "completed_lower_secondary", 2: "dropped_out"}
df_clean["school_complete3_lbl"] = df_clean["school_complete3"].map(label_map).astype("string")


# =========================
# 9. Marital variables — FORCE NA in 2023 (no Q1101 age at first marriage)
# =========================
df_clean["age_marry"] = pd.Series(pd.NA, index=df_clean.index, dtype="Float64")
df_clean["married_by19"] = pd.Series(pd.NA, index=df_clean.index, dtype="Float64")
df_clean["marriage_timing"] = pd.Series(pd.NA, index=df_clean.index, dtype="object")


# =========================
# 10. Birth year/month & pregnancy timing
# =========================
df_clean.loc[df_clean["born_year"].isna(), "born_year"] = SURVEY_YEAR - df_clean["age_completed"]
df_clean.loc[df_clean["born_month"].isna(), "born_month"] = 7

df_clean["born_month"] = pd.to_numeric(df_clean["born_month"], errors="coerce")
df_clean["born_year"] = pd.to_numeric(df_clean["born_year"], errors="coerce")

df_clean.loc[~df_clean["born_month"].between(1, 12), "born_month"] = np.nan
df_clean.loc[~df_clean["born_year"].between(SURVEY_YEAR - 24, SURVEY_YEAR - 10), "born_year"] = np.nan
df_clean["born_month"] = df_clean["born_month"].fillna(7)

df_clean["age_months"] = (SURVEY_YEAR - df_clean["born_year"]) * 12 + (7 - df_clean["born_month"])

df_clean["age_preg"] = pd.to_numeric(df_clean["age_preg"], errors="coerce")
df_clean.loc[~df_clean["age_preg"].between(10, 24), "age_preg"] = np.nan
df_clean.loc[(df_clean["been_preg"] == 1) & (df_clean["age_preg"].isna()), "age_preg"] = df_clean["age_completed"] - 1


# =========================
# 11. Consistency: sex_age <= age_preg, etc.
# =========================
df_clean["sex_age"] = pd.to_numeric(df_clean["sex_age"], errors="coerce")

df_clean.loc[(df_clean["been_preg"] == 0) & (df_clean["age_preg"].notna()), "been_preg"] = 1
df_clean.loc[~df_clean["sex_age"].between(5, 24), "sex_age"] = np.nan
df_clean.loc[(df_clean["been_preg"] == 1) & (df_clean["sex_age"].isna()), "sex_age"] = df_clean["age_preg"]
df_clean.loc[(df_clean["been_preg"] == 1) & (df_clean["sex_age"] > df_clean["age_preg"]), "sex_age"] = df_clean["age_preg"]
df_clean.loc[(df_clean["been_preg"] == 1) & (df_clean["age_preg"] > df_clean["age_completed"]), "age_preg"] = df_clean["age_completed"]

df_clean = df_clean[(df_clean["age_preg"].isna()) | (df_clean["age_preg"] <= 25)].copy()

df_clean.loc[df_clean["been_preg"] == 1, "diff"] = df_clean["age_months"] / 12 - df_clean["age_preg"]
df_clean.loc[df_clean["diff"] < -0.5, "diff"] = np.nan

df_clean["ado_preg"] = ((df_clean["been_preg"] == 1) & (df_clean["age_preg"] <= 19)).fillna(False).astype("int8")

mask_fix = (
    (df_clean["been_preg"] == 1)
    & df_clean["sex_age"].notna()
    & df_clean["age_preg"].notna()
    & (df_clean["sex_age"] > df_clean["age_preg"])
)
df_clean.loc[mask_fix, "sex_age"] = df_clean["age_preg"]



# =========================
# 12. Age cohort
# =========================
df_clean["age_cohort"] = np.where(df_clean["age_completed"] <= 14, "10–14", "15–19")


# =========================
# 13. Wealth PCA & tertiles
# =========================
asset_vars = [
    "radio", "tv_set", "bicycle", "motorcycle", "own_home",
    "cell_phone", "reg_phone", "computer", "income_busin",
    "bath_room", "run_water", "electricity", "car", "generator", "solar"
]

df_assets = df_clean[asset_vars].replace(98, np.nan).copy()
df_assets = df_assets.applymap(lambda x: 1 if x == 1 else 0).fillna(0)

scaler = StandardScaler()
assets_scaled = scaler.fit_transform(df_assets)

pca = PCA(n_components=1)
wealth_index = pca.fit_transform(assets_scaled)
df_clean["wealth_index"] = wealth_index[:, 0]

w = df_clean["wealth_index"]
mask = w.notna()
ranks = w[mask].rank(method="first")
labels = ["Low", "Medium", "High"]
tertiles = pd.qcut(ranks, 3, labels=labels)

df_clean["wealth_tertile"] = pd.NA
df_clean.loc[mask, "wealth_tertile"] = tertiles.astype(str)
df_clean["wealth_tertile"] = pd.Categorical(df_clean["wealth_tertile"], categories=labels, ordered=True)


# =========================
# 14. Final export aligned with 2018 schema
# =========================
FEATURES = [
    "ado_preg",
    "been_preg",
    "age_completed",
    "age_cohort",
    "age_preg",
    "sex_age",
    "will_sex_binary",
    "do_anything_binary",
    "age_marry",
    "married_by19",
    "marriage_timing",
    "school_complete3_lbl",
    "years_school",
    "level_scol_recode",
    "edu_bucket_highest",
    "person_sex_group",
    "male_condom",
    "female_condom",
    "iud_coil",
    "avoid_other",
    "pill",
    "withdrawal",
    "implant",
    "often_usecondom",
    "sex_active_12m",
    "condom_use_ord",
    "condom_use_ord_active",
    "wealth_tertile",
]

cols = [c for c in FEATURES if c in df_clean.columns]
df_export_2023 = df_clean[cols].copy()
df_export_2023["year"] = SURVEY_YEAR

df_export_2023.to_csv("./data/processed_df_2023_aligned.csv", index=False)
print("\nExported 2023 file with columns:")
print(df_export_2023.info())
print(df_export_2023.head())


Current directory: /Users/nataschajademinnitt/Documents/5_data/teenage_pregnancy
After basic filtering (ever-pregnancy asked): (3307, 65)
life_sex
1    3307
Name: count, dtype: Int64
scol_status
OUT OF SCHOOL    2066
IN SCHOOL        1241
Name: count, dtype: Int64

Exported 2023 file with columns:
<class 'pandas.core.frame.DataFrame'>
Index: 3307 entries, 0 to 5456
Data columns (total 29 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                 --------------  -----   
 0   ado_preg               3307 non-null   int8    
 1   been_preg              3307 non-null   Int64   
 2   age_completed          3307 non-null   float64 
 3   age_cohort             3307 non-null   object  
 4   age_preg               1614 non-null   float64 
 5   sex_age                3293 non-null   float32 
 6   will_sex_binary        3307 non-null   Int64   
 7   do_anything_binary     3280 non-null   Float64 
 8   age_marry              0 non-null      Float64 
 9   married_by1

# Quality checks

In [3]:
import pandas as pd
import numpy as np

df18 = pd.read_csv("./data/processed_df_2018_aligned.csv")
df23 = pd.read_csv("./data/processed_df_2023_aligned.csv")

cols18 = set(df18.columns)
cols23 = set(df23.columns)

missing_in_23 = sorted(cols18 - cols23)
extra_in_23   = sorted(cols23 - cols18)

print("Missing in 2023 (present in 2018):", missing_in_23)
print("Extra in 2023 (not in 2018):", extra_in_23)

def missingness_table(df, name):
    out = pd.DataFrame({
        "n": len(df),
        "non_null": df.notna().sum(),
        "pct_missing": (df.isna().mean() * 100).round(1),
        "n_unique": df.nunique(dropna=True)
    }).sort_values("pct_missing", ascending=False)
    print(f"\n=== Missingness: {name} ===")
    return out

m18 = missingness_table(df18, "2018")
m23 = missingness_table(df23, "2023")

display(m18.head(15))
display(m23.head(15))

all_missing_23 = df23.columns[df23.isna().all()].tolist()
near_missing_23 = df23.columns[(df23.isna().mean() > 0.95)].tolist()

print("All-missing columns in 2023:", all_missing_23)
print(">=95% missing columns in 2023:", near_missing_23)

all_missing_18 = df18.columns[df18.isna().all()].tolist()
near_missing_18 = df18.columns[(df18.isna().mean() > 0.95)].tolist()

print("All-missing columns in 2018:", all_missing_18)
print(">=95% missing columns in 2018:", near_missing_18)

common = sorted(cols18 & cols23)

type_mismatches = []
for c in common:
    t18 = str(df18[c].dtype)
    t23 = str(df23[c].dtype)
    if t18 != t23:
        type_mismatches.append((c, t18, t23))

type_mismatches[:20], len(type_mismatches)

cat_like = ["age_cohort","marriage_timing","school_complete3_lbl","edu_bucket_highest","person_sex_group","wealth_tertile"]

for c in cat_like:
    if c in df18.columns and c in df23.columns:
        v18 = set(df18[c].dropna().astype(str).unique())
        v23 = set(df23[c].dropna().astype(str).unique())
        print(f"\n{c}")
        print("  only in 2018:", sorted(v18 - v23))
        print("  only in 2023:", sorted(v23 - v18))

def sanity_checks(df, year_label=""):
    problems = {}

    # ado_preg should imply been_preg==1
    if {"ado_preg","been_preg"}.issubset(df.columns):
        problems["ado_preg_but_not_been_preg"] = df[(df["ado_preg"]==1) & (df["been_preg"]!=1)].shape[0]

    # if been_preg==0 then age_preg should be missing
    if {"been_preg","age_preg"}.issubset(df.columns):
        problems["age_preg_present_but_been_preg_0"] = df[(df["been_preg"]==0) & (df["age_preg"].notna())].shape[0]

    # sex_age should not exceed age_preg among pregnant
    if {"been_preg","sex_age","age_preg"}.issubset(df.columns):
        problems["sex_age_gt_age_preg_when_preg"] = df[(df["been_preg"]==1) & (df["sex_age"].notna()) & (df["age_preg"].notna()) & (df["sex_age"]>df["age_preg"])].shape[0]

    # age bounds
    if "age_preg" in df.columns:
        problems["age_preg_outside_10_24"] = df[(df["age_preg"].notna()) & ((df["age_preg"]<10) | (df["age_preg"]>24))].shape[0]
    if "sex_age" in df.columns:
        problems["sex_age_outside_5_24"] = df[(df["sex_age"].notna()) & ((df["sex_age"]<5) | (df["sex_age"]>24))].shape[0]

    print(f"\n=== Sanity checks {year_label} ===")
    for k,v in problems.items():
        print(f"{k}: {v}")

sanity_checks(df18, "2018")
sanity_checks(df23, "2023")

common = sorted(set(df18.columns) & set(df23.columns))

report = pd.DataFrame({
    "missing_%_2018": (df18[common].isna().mean()*100).round(1),
    "missing_%_2023": (df23[common].isna().mean()*100).round(1),
    "n_unique_2018": df18[common].nunique(dropna=True),
    "n_unique_2023": df23[common].nunique(dropna=True),
})

report["missing_diff_23_minus_18"] = (report["missing_%_2023"] - report["missing_%_2018"]).round(1)
report = report.sort_values("missing_diff_23_minus_18", ascending=False)

display(report.head(25))



Missing in 2023 (present in 2018): []
Extra in 2023 (not in 2018): []

=== Missingness: 2018 ===

=== Missingness: 2023 ===


,n,non_null,pct_missing,n_unique
avoid_other,8076,765,90.50,2
implant,8076,798,90.10,2
female_condom,8076,811,90.00,2
iud_coil,8076,805,90.00,2
withdrawal,8076,832,89.70,2
pill,8076,966,88.00,2
age_marry,8076,1375,83.00,14
condom_use_ord,8076,1792,77.80,3
male_condom,8076,2011,75.10,2
age_preg,8076,2347,70.90,14


,n,non_null,pct_missing,n_unique
marriage_timing,3307,0,100.00,0
age_marry,3307,0,100.00,0
married_by19,3307,0,100.00,0
condom_use_ord,3307,926,72.00,3
withdrawal,3307,1284,61.20,2
avoid_other,3307,1284,61.20,2
iud_coil,3307,1284,61.20,1
female_condom,3307,1284,61.20,2
male_condom,3307,1284,61.20,2
implant,3307,1284,61.20,2


All-missing columns in 2023: ['age_marry', 'married_by19', 'marriage_timing']
>=95% missing columns in 2023: ['age_marry', 'married_by19', 'marriage_timing']
All-missing columns in 2018: []
>=95% missing columns in 2018: []

age_cohort
  only in 2018: []
  only in 2023: []

marriage_timing
  only in 2018: ['marry_after_preg', 'marry_before_preg', 'marry_never_preg', 'never_marry_or_preg', 'preg_never_marry']
  only in 2023: []

school_complete3_lbl
  only in 2018: []
  only in 2023: []

edu_bucket_highest
  only in 2018: ['less_than_UPE']
  only in 2023: []

person_sex_group
  only in 2018: []
  only in 2023: []

wealth_tertile
  only in 2018: []
  only in 2023: []

=== Sanity checks 2018 ===
ado_preg_but_not_been_preg: 0
age_preg_present_but_been_preg_0: 0
sex_age_gt_age_preg_when_preg: 0
age_preg_outside_10_24: 0
sex_age_outside_5_24: 0

=== Sanity checks 2023 ===
ado_preg_but_not_been_preg: 0
age_preg_present_but_been_preg_0: 0
sex_age_gt_age_preg_when_preg: 0
age_preg_outside_10_24

,missing_%_2018,missing_%_2023,n_unique_2018,n_unique_2023,missing_diff_23_minus_18
married_by19,0.00,100.00,2,0,100.00
marriage_timing,0.00,100.00,5,0,100.00
age_marry,83.00,100.00,14,0,17.00
edu_bucket_highest,1.70,1.80,5,4,0.10
ado_preg,0.00,0.00,2,2,0.00
year,0.00,0.00,1,1,0.00
wealth_tertile,0.00,0.00,3,3,0.00
sex_active_12m,0.00,0.00,2,2,0.00
age_cohort,0.00,0.00,2,2,0.00
years_school,0.00,0.00,15,5,0.00
